# Tugas 1 - Klasifikasi Wine Quality dengan KNN
  
| Nama              | NRP        |
|-------------------|------------|
| Benedictus Ryu Gunawan | 5054251001|

## Deskripsi Tugas
Pada tugas ini, kita akan menggunakan Wine Quality Dataset. Dataset bisa diakses melalui link berikut:\
🔗 https://www.kaggle.com/datasets/yasserh/wine-quality-dataset

Tujuan utama dari tugas ini adalah membangun model K-Nearest Neighbors (KNN) untuk mengklasifikasikan kualitas wine. 

Langkah-langkah yang harus dilakukan antara lain:
1. Persiapan Dataset & Eksplorasi Awal

- Memuat dataset, melihat struktur data, dan distribusi label.

2. Preprocessing 
- Memproses data agar siap untuk digunakan dalam membangun model.

3. Eksperimen Model KNN
- Bangun model KNN dengan mencoba beberapa nilai k (misalnya 3, 5, dan 7 --> BEBAS) serta dua metric jarak (seperti Euclidean dan Manhattan).
- Eksperimen ini bertujuan untuk membandingkan performa KNN dengan parameter yang berbeda.

4. Evaluasi Model
- Hitung metrik evaluasi seperti Accuracy, Precision, Recall, F1-Score, serta visualisasikan Confusion Matrix.

5. Analisis & Kesimpulan
- Bandingkan hasil antar eksperimen yang telah dilakukan dan berikan kesimpulan.


# 1. Persiapan Dataset & Eksplorasi Awal

In [ ]:
# Import library yang dibutuhkan (pandas, numpy, matplotlib, dll) 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [ ]:
#  Load Dataset dan menampilkan informasi dasar tentang dataset, seperti jumlah baris dan kolom, tipe data, jumlah nilai yang hilang, dan informasi lainnya yang dibutuhkan.

data = pd.read_csv('../data/WineQT.csv')
data.index = data['Id']
data.drop('Id', axis=1, inplace=True)
experiment_dataset = [data]
data.head()
data.info()
data.describe()


In [ ]:
import seaborn as sns

# Tampilkan Korelasi Matrix
plt.figure(figsize=(10, 8))
sns.heatmap(data.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)

plt.title('Korelasi Matrix')
plt.show()

display(pd.DataFrame(data.corr()).to_string())

In [ ]:
# Tampilkan distribusi label (quality) dalam bentuk visualisasi, misalnya menggunakan histogram atau bar plot.
import seaborn as sns

frequency = data['quality'].value_counts().sort_index()

plt.figure(figsize=(8, 6))
plt.hist(data['quality'], bins=np.arange(data['quality'].min(), data['quality'].max() + 1) - 0.5, edgecolor='black',color='skyblue')
plt.title('Distribusi Label (Quality)')

for i in range(len(frequency)):
    plt.text(frequency.index[i], frequency.values[i], str(frequency.values[i]), ha='center', va='bottom')

plt.xlabel('Quality')
plt.ylabel('Jumlah')
plt.show()

In [ ]:
from umap import UMAP

dimensionality_reduction = UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=42)
data_reduced = dimensionality_reduction.fit_transform(data.drop('quality', axis=1))
plt.figure(figsize=(8, 6))
plt.scatter(data_reduced[:, 0], data_reduced[:, 1], c=data['quality'], cmap='viridis', edgecolor='k', s=50)
plt.colorbar(label='Quality')
plt.title('UMAP Dimensionality Reduction')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.show()

In [ ]:
def plot_feature_distribution(data, feature):
    n = len(feature)
    fig,ax = plt.subplots(n//3 + (1 if n % 3 else 0), 3, figsize=(10, 12))
    for i, col in enumerate(feature):
        sns.histplot(data[col], kde=True, ax=ax[i//3, i%3])
        ax[i//3, i%3].set_title(f'Distribution of {col}')
    plt.tight_layout()

plot_feature_distribution(data, data.columns)

In [ ]:
# Skewness dan Kurtosis
skewness = data.drop('quality', axis=1).skew().sort_values(ascending=False)
kurtosis = data.drop('quality', axis=1).kurtosis().sort_values(ascending=False)
print("Skewness:\n", skewness)
print("\nKurtosis:\n", kurtosis)

Banyak dari feature mengalami right-skewed (Artinya secara logis, features scaling dan juga transformasi logaritmik akan berperan penting dalam perhitungan)

# 2. Preprocessing




## 2.1 Missing & Duplicate Checking

In [ ]:
# NaN handling (drop, impute, etc.), duplicate handling, dan lain-lain.

import missingno as msno

fig,ax = plt.subplots(1,2,figsize=(10,6))
ax[0].set_title('Missing Values Matrix')
msno.matrix(data, ax=ax[0])
ax[1].set_title('Missing Values Heatmap')
msno.heatmap(data, ax=ax[1])

print(f"Number of Duplicated Rows: {data.duplicated().sum()}")


Tidak ada duplicate dalam dataset ini sehingga, tidak perlu adanya penanganan pada missing dan duplicate values

## 2.2 See the outlier

In [ ]:
# Periksa Outliers (jika diperlukan), misalnya menggunakan boxplot atau metode statistik lainnya. (Bisa dilakukan eksperimen dengan dan tanpa outlier untuk melihat perbedaan performa model)
def plot_boxplots(data):
    fig,ax = plt.subplots(4,3,figsize=(15,20))
    for i, column in enumerate(data.columns[:-1]):
        sns.boxplot(data[column], ax=ax[i//3, i%3])
        ax[i//3, i%3].set_title(f'Boxplot of {column}')

plot_boxplots(data)

In [ ]:
# Clip Outlier Handling

outlier_features = [
    "chlorides",
    "residual sugar",
    "sulphates",
    "total sulfur dioxide"
]

df_clip = data.copy()

for col in outlier_features:
    lower = df_clip[col].quantile(0.01)
    upper = df_clip[col].quantile(0.99)

    df_clip[col] = df_clip[col].clip(lower, upper)

experiment_dataset.append(df_clip)
plot_boxplots(df_clip)

## 2.3 Feature Engineering

In [ ]:
def feature_engineering(data):
    data['total_acidity'] = data['fixed acidity'] + data['volatile acidity']
    data['density_alcohol_ratio'] = data['density'] / data['alcohol']
    eps = 1e-6

    data["total_acidity"] = (
        data["fixed acidity"]
        + data["volatile acidity"]
        + data["citric acid"]
    )

    data["volatile_fixed_ratio"] = (
        data["volatile acidity"] / (data["fixed acidity"] + eps)
    )

    data["free_total_sulfur_ratio"] = (
        data["free sulfur dioxide"]
        / (data["total sulfur dioxide"] + eps)
    )

    data["bound_sulfur_dioxide"] = (
        data["total sulfur dioxide"]
        - data["free sulfur dioxide"]
    )

    data["alcohol_density_ratio"] = (
        data["alcohol"] / (data["density"] + eps)
    )

    data["sugar_alcohol_ratio"] = (
        data["residual sugar"] / (data["alcohol"] + eps)
    )

    data["sulphates_chlorides_ratio"] = (
        data["sulphates"] / (data["chlorides"] + eps)
    )

    data["acid_pH_interaction"] = (
        data["fixed acidity"] * data["pH"]
    )

    engineered_features = [
        "total_acidity",
        "volatile_fixed_ratio",
        "free_total_sulfur_ratio",
        "bound_sulfur_dioxide",
        "alcohol_density_ratio",
        "sugar_alcohol_ratio",
        "sulphates_chlorides_ratio",
        "acid_pH_interaction"
    ]

    data[[f"log_{col}" for col in engineered_features]] = (
        data[engineered_features]
        .clip(lower=0)
        .apply(np.log1p)
        .set_axis(
            [f"log_{col}" for col in engineered_features],
            axis=1
        )
    )
        
    return data

df_fe = feature_engineering(df_clip.copy())
experiment_dataset.append(df_fe)

## 2.4 Feature Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

df_scaled = df_fe.copy()
target = df_scaled['quality']
df_scaled.drop('quality', axis=1, inplace=True)
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df_scaled)
df_scaled = pd.DataFrame(scaled_features, columns=df_scaled.columns)
df_scaled['quality'] = target.values

experiment_dataset.append(df_scaled)
df_scaled

In [ ]:
from sklearn.preprocessing import MinMaxScaler

df_minmax = df_fe.copy()
target = df_minmax['quality']
df_minmax.drop('quality', axis=1, inplace=True)
minmax_scaler = MinMaxScaler()
minmax_scaled_features = minmax_scaler.fit_transform(df_minmax)
df_minmax = pd.DataFrame(minmax_scaled_features, columns=df_minmax.columns)
df_minmax['quality'] = target.values

experiment_dataset.append(df_minmax)
df_minmax

In [ ]:
from sklearn.preprocessing import RobustScaler

df_robust = df_fe.copy()
target = df_robust['quality']
df_robust.drop('quality', axis=1, inplace=True)
robust_scaler = RobustScaler()
robust_scaled_features = robust_scaler.fit_transform(df_robust)
df_robust = pd.DataFrame(robust_scaled_features, columns=df_robust.columns)
df_robust['quality'] = target.values

experiment_dataset.append(df_robust)
df_robust

## 2.5 SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

X_smote_base = df_scaled.drop('quality', axis=1)
y_smote_base = df_scaled['quality']
X_train_smote_base, X_test_smote_holdout, y_train_smote_base, y_test_smote_holdout = train_test_split(
    X_smote_base, y_smote_base, test_size=0.2, random_state=42, stratify=y_smote_base
)

smote = SMOTE(random_state=42, k_neighbors=1)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_smote_base, y_train_smote_base)

df_smote_train_resampled = pd.DataFrame(X_train_resampled, columns=X_smote_base.columns)
df_smote_train_resampled['quality'] = y_train_resampled.values
print(f"Train before SMOTE: {y_train_smote_base.value_counts().sort_index().to_dict()}")
print(f"Train after SMOTE: {y_train_resampled.value_counts().sort_index().to_dict()}")
print(f"Holdout (untouched): {y_test_smote_holdout.value_counts().sort_index().to_dict()}")


# 3. Eksperimen Model KNN


In [ ]:
# Create KNN From scratch

class KNN:
    def __init__(self, k=3, distances_technique='euclidean', weights='uniform'):
        self.k = k
        self.distances_technique = distances_technique
        self.weights = weights

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        predictions = [self._predict(x) for x in X]
        return np.array(predictions)

    def _predict(self, x):
        # Hitung jarak antara x dan semua titik data dalam X_train\
        if self.distances_technique == 'euclidean':
            distances = np.linalg.norm(self.X_train - x, axis=1)
        elif self.distances_technique == 'manhattan':
            distances = np.sum(np.abs(self.X_train - x), axis=1)
        elif self.distances_technique == 'minkowski':
            p = 3  # Contoh nilai p untuk Minkowski
            distances = np.sum(np.abs(self.X_train - x) ** p, axis=1) ** (1/p)
        else:
            raise ValueError("Unknown distance technique")
        

        # Urutkan jarak dan ambil indeks dari k tetangga terdekat
        k_indices = np.argsort(distances)[:self.k]

        if self.weights == 'distance':
            # Ambil jarak dari k tetangga terdekat
            k_nearest_distances = distances[k_indices]
            # Hitung bobot sebagai invers dari jarak
            weights = 1 / (k_nearest_distances + 1e-5)  # Tambahkan epsilon untuk menghindari pembagian dengan nol
            # Ambil label dari k tetangga terdekat
            k_nearest_labels = [self.y_train[i] for i in k_indices]
            # Hitung label yang paling umum dengan mempertimbangkan bobot
            label_weights = {}
            for label, weight in zip(k_nearest_labels, weights):
                if label in label_weights:
                    label_weights[label] += weight
                else:
                    label_weights[label] = weight
            most_common = max(label_weights, key=label_weights.get)
        elif self.weights == 'uniform':
            # Ambil label dari k tetangga terdekat
            k_nearest_labels = [self.y_train[i] for i in k_indices]
            # Hitung label yang paling umum
            most_common = max(set(k_nearest_labels), key=k_nearest_labels.count)
        else:
            raise ValueError("Unknown weights technique")


        return most_common

model = KNN(k=3)
model

In [ ]:
from dataclasses import dataclass

@dataclass
class ModelEvaluation:
    data_version: str
    model_version: str
    accuracy: float
    precision: float
    recall: float
    f1_score: float

# 4. Evaluasi Model


## 4.1 Evaluasi KNN From Scratch

In [ ]:
# Testing
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(df_scaled.drop('quality', axis=1), df_scaled['quality'], test_size=0.2, random_state=42, stratify=df_scaled['quality'])

In [ ]:
# Tampilkan evaluasi model KNN dengan metriks evaluasi yang sesuai dengan tipe masalah klasifikasi

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(y_true, y_pred):
    print("Confusion Matrix:")
    display(confusion_matrix(y_true, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred))
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred, average='weighted'):.4f}")
    print(f"Recall: {recall_score(y_true, y_pred, average='weighted'):.4f}")
    print(f"F1 Score: {f1_score(y_true, y_pred, average='weighted'):.4f}")

In [ ]:
model.fit(X_train.values, y_train.values)
evaluate_model(y_test.values, model.predict(X_test.values))

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier(n_neighbors=3)
model.fit(X_train.values, y_train.values)
model.predict(X_test.values)
evaluate_model(y_test.values, model.predict(X_test.values))

Hasil from scratch sama hasil dari model bawaan sudah sama, karena kedua model tersebut sebenarnya adalah lazy learners

## 4.2 Evaluasi Dataset

In [ ]:
def evaluate_dataset(X_train, X_test, y_train, y_test, model, data_version, model_version):
    model.fit(X_train.values, y_train.values)
    y_pred = model.predict(X_test.values)
    
    accuracy = accuracy_score(y_test.values, y_pred)
    precision = precision_score(y_test.values, y_pred, average='weighted')
    recall = recall_score(y_test.values, y_pred, average='weighted')
    f1 = f1_score(y_test.values, y_pred, average='weighted')
    
    
    return accuracy, precision, recall, f1

In [ ]:
from sklearn.model_selection import KFold, cross_val_score
from imblearn.over_sampling import SMOTE

def evaluate_all_datasets(model):
    evaluations = []
    for dataset in experiment_dataset:
        X_train, X_test, y_train, y_test = train_test_split(dataset.drop('quality', axis=1), dataset['quality'], test_size=0.2, random_state=42, stratify=dataset['quality'])
        KF = KFold(n_splits=5, shuffle=True, random_state=42)
        eval = ModelEvaluation(data_version="baseline" if dataset.equals(data) else  "clip" if dataset.equals(df_clip) else "fe" if dataset.equals(df_fe) else "standard" if dataset.equals(df_scaled) else "minmax" if dataset.equals(df_minmax) else "robust", model_version="KNN", accuracy=0, precision=0, recall=0, f1_score=0)
        for train_index, test_index in KF.split(X_train):
            X_train_fold, X_test_fold = X_train.iloc[train_index], X_train.iloc[test_index]
            y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
            model.fit(X_train_fold.values, y_train_fold.values)
            y_pred_fold = model.predict(X_test_fold.values)

            accuracy, precision, recall, f1 = evaluate_dataset(X_train_fold, X_test_fold, y_train_fold, y_test_fold, model, eval.data_version, eval.model_version)
            eval.accuracy += accuracy
            eval.precision += precision
            eval.recall += recall
            eval.f1_score += f1
        eval.accuracy /= KF.get_n_splits()
        eval.precision /= KF.get_n_splits()
        eval.recall /= KF.get_n_splits()
        eval.f1_score /= KF.get_n_splits()
        evaluations.append(eval)

    return evaluations

def evaluate_with_train_only_smote(model, dataset, label):
    # CORRECT SMOTE: split first, then SMOTE each train fold only. Test fold stays untouched.
    # k_neighbors=1 because quality=3/8 have <6 samples per fold (default k=5 would crash).
    X_train, X_test, y_train, y_test = train_test_split(dataset.drop('quality', axis=1), dataset['quality'], test_size=0.2, random_state=42, stratify=dataset['quality'])
    KF = KFold(n_splits=5, shuffle=True, random_state=42)
    smote = SMOTE(random_state=42, k_neighbors=1)
    eval = ModelEvaluation(data_version=label, model_version="KNN", accuracy=0, precision=0, recall=0, f1_score=0)
    for train_index, test_index in KF.split(X_train):
        X_tr, X_te = X_train.iloc[train_index], X_train.iloc[test_index]
        y_tr, y_te = y_train.iloc[train_index], y_train.iloc[test_index]
        X_tr_res, y_tr_res = smote.fit_resample(X_tr, y_tr)
        model.fit(np.asarray(X_tr_res), np.asarray(y_tr_res))
        y_pred = model.predict(np.asarray(X_te))
        eval.accuracy += accuracy_score(np.asarray(y_te), y_pred)
        eval.precision += precision_score(np.asarray(y_te), y_pred, average='weighted', zero_division=0)
        eval.recall += recall_score(np.asarray(y_te), y_pred, average='weighted', zero_division=0)
        eval.f1_score += f1_score(np.asarray(y_te), y_pred, average='weighted', zero_division=0)
    eval.accuracy /= KF.get_n_splits()
    eval.precision /= KF.get_n_splits()
    eval.recall /= KF.get_n_splits()
    eval.f1_score /= KF.get_n_splits()
    return eval

evaluations = evaluate_all_datasets(model)
evaluations.append(evaluate_with_train_only_smote(model, df_scaled, "smote"))

In [ ]:
plt.figure(figsize=(10, 6))
data_versions = [e.data_version for e in evaluations]
accuracies = [e.accuracy for e in evaluations]
plt.bar(data_versions, accuracies, color='skyblue')
plt.title('Model Accuracy Comparison Across Different Data Versions')
for i, v in enumerate(accuracies):
    plt.text(i, v + 0.005, f"{v:.4f}", ha='center', va='bottom')
plt.xlabel('Data Version')
plt.ylabel('Accuracy')
plt.show()

## 4.3 Evaluasi Hyperparameter Model

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df_minmax.drop('quality', axis=1), df_minmax['quality'], test_size=0.2, random_state=42, stratify=df_minmax['quality'])

In [ ]:
from sklearn.model_selection import GridSearchCV

params = {
    'n_neighbors': [i for i in range(1, 20,2)],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

grid_search = GridSearchCV(model, params, cv=5, scoring='accuracy')
grid_search.fit(X_train.values, y_train.values)

print("Best parameters found: ", grid_search.best_params_)
print("Best accuracy found: ", grid_search.best_score_)


In [ ]:
# Convert all GridSearchCV results to DataFrame
results = pd.DataFrame(grid_search.cv_results_)

# Convert parameter columns to convenient types
results["param_n_neighbors"] = results["param_n_neighbors"].astype(int)

# Sort for consistent plotting
results = results.sort_values(
    by=["param_metric", "param_weights", "param_n_neighbors"]
)

# Plot all parameter combinations
fig, ax = plt.subplots(figsize=(12, 7))

for (metric, weight), group in results.groupby(
    ["param_metric", "param_weights"]
):
    group = group.sort_values("param_n_neighbors")

    ax.plot(
        group["param_n_neighbors"],
        group["mean_test_score"],
        marker="o",
        linewidth=2,
        label=f"{metric} | {weight}"
    )

    # CV standard deviation band
    ax.fill_between(
        group["param_n_neighbors"],
        group["mean_test_score"] - group["std_test_score"],
        group["mean_test_score"] + group["std_test_score"],
        alpha=0.12
    )

# Mark the best result
best_index = results["mean_test_score"].idxmax()
best_result = results.loc[best_index]

ax.scatter(
    best_result["param_n_neighbors"],
    best_result["mean_test_score"],
    color="red",
    marker="*",
    s=250,
    zorder=5,
    label=(
        f"Best: k={best_result['param_n_neighbors']}, "
        f"{best_result['param_metric']}, "
        f"{best_result['param_weights']}"
    )
)

ax.set_title("KNN Grid Search Cross-Validation Results")
ax.set_xlabel("Number of Neighbors")
ax.set_ylabel("Mean CV Accuracy")
ax.set_xticks(sorted(results["param_n_neighbors"].unique()))
ax.grid(alpha=0.3)
ax.legend(title="Metric | Weight", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

# 5. Analisis

Berdasakan hasil dari percobaan diatas, dapai disimpulkan model sangatlah bergantung pada parameter yang diberikan pada model tersebut. Selain itu, model juga sangat berkegantungan dengan scaling feature mengingat KNN menggunakna representatif jarak sebagai teknik inference. Jika tidak meggunakan kedua teknik tersebut, kemampuan model dalma memahami data dapat berkurang.

Dalam dataset wine ini, terdapat beberapa feature dengan outlier seperti, `chlorides`, `residual sugar` `sulphates`, `total sulfur dioxide`. Penanganan yang dilakukan menggunakan clip dan transformasi logaritmik pada feature engeeringnya

# 6. Kesimpulan dan Saran

### Kesimpulan

1. Model KNN from-scratch dan `sklearn.neighbors.KNeighborsClassifier` menghasilkan performa yang setara (akurasi ~0,57–0,59 pada `df_scaled` dengan k=3). Hal ini sesuai ekspektasi karena keduanya adalah lazy learner berbasis jarak.
2. KNN sangat sensitif terhadap skala fitur. Versi data yang di-scaling (`standard`/`minmax`/`robust`) mengungguli data mentah, karena fitur seperti `total sulfur dioxide` dan `alcohol` memiliki rentang yang jauh berbeda sehingga mendominasi perhitungan jarak jika tidak dinormalisasi.
3. Penanganan outlier dengan clipping kuantil 1%–99% pada `chlorides`, `residual sugar`, `sulphates`, `total sulfur dioxide` serta feature engineering (rasio-rasio seperti `volatile_fixed_ratio`, `alcohol_density_ratio`, dan versi `log1p`-nya) membuat distribusi lebih stabil dan membantu KNN.
4. SMOTE hanya valid bila diterapkan pada data train saja (split dulu, baru `fit_resample` pada train / tiap train-fold). SMOTE sebelum split menyebabkan leakage dan membuat skor terlihat optimistis, terutama untuk KNN yang menghafal ketetanggaan lokal. Dengan SMOTE train-only (`k_neighbors=1` karena kelas 3/8 hanya punya <6 sampel per fold), recall kelas minoritas membaik tanpa mencemari test fold yang tetap orisinal.
5. Tuning hiperparameter berpengaruh besar. `GridSearchCV` 5-fold pada `df_minmax` menemukan konfigurasi terbaik `metric=manhattan, n_neighbors=12, weights=distance` dengan akurasi CV ~0,644, lebih baik dari baseline k=3 (`euclidean`, `uniform`). Bobot berbasis jarak (`distance`) membantu karena tetangga yang lebih dekat diberi pengaruh lebih besar.
6. Keterbatasan utama adalah ketidakseimbangan kelas: `quality=5` dan `6` mendominasi, sedangkan `quality=3, 4, 8` hanya punya belasan sampel. Akibatnya precision/recall kelas minoritas mendekati 0 meskipun akurasi global ~0,6. Confusion matrix menunjukkan banyak sampel kelas 4/7 tertukar ke kelas 5/6 yang berdekatan.

### Saran

1. Bungkus preprocessing + SMOTE + KNN dalam `imblearn.pipeline.Pipeline` agar `fit` (scaler, clipping, SMOTE) hanya dipelajari dari train-fold. Gunakan `StratifiedKFold` agar proporsi kelas minoritas terjaga di tiap fold.
2. Untuk eksperimen lanjutan: uji SMOTE vs class-weight vs undersampling, variasi `k_neighbors` SMOTE, serta seleksi fitur (misalnya berbasis korelasi pada heatmap) untuk mengurangi noise dimensi hasil feature engineering.
3. Pertimbangkan penggabungan kelas langka (misalnya 3+4 sebagai `low`, 7+8 sebagai `high`) atau evaluasi dengan metrik yang adil untuk imbalance seperti macro-F1 / balanced accuracy, bukan hanya akurasi.
4. Bandingkan KNN dengan model yang lebih robust terhadap imbalance dan outlier (misalnya Random Forest atau Logistic Regression berpenalti) sebagai pembanding.